# Stack Shape Explorer
Scans all subjects in `myosegmenTUM` and reports the shape, voxel spacing,
and origin of every image stack and its matching ground-truth mask.

In [1]:
import glob, os, re
import numpy as np
import pandas as pd
import SimpleITK as sitk
import ipywidgets as widgets
from IPython.display import display

DATA_ROOT = r'C:\Projects\dissector\eval_notebooks\myosegmenTUM'

In [2]:
# ── Collect metadata for every stack ─────────────────────────────────────────
def sitk_meta(path):
    img = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(img)
    sp  = img.GetSpacing()          # (x, y, z) in mm
    return {
        'slices': arr.shape[0],
        'height': arr.shape[1],
        'width':  arr.shape[2],
        'shape':  str(arr.shape),
        'sp_x_mm': round(sp[0], 3),
        'sp_y_mm': round(sp[1], 3),
        'sp_z_mm': round(sp[2], 3),
    }

rows = []

for subj_dir in sorted(glob.glob(os.path.join(DATA_ROOT, '*'))):
    subject = os.path.basename(subj_dir)
    subj_type = 'P' if subject.startswith('P') else 'HV'

    # Water stacks
    water_files = sorted(glob.glob(
        os.path.join(subj_dir, 'ImageData', f'{subject}_WATER',
                     f'{subject}_WATER_stack*.nii')))

    for wpath in water_files:
        m = re.search(r'stack(\d+)', os.path.basename(wpath))
        if not m:
            continue
        stack_n = int(m.group(1))

        row = {'subject': subject, 'type': subj_type, 'stack': stack_n}

        # Water image
        try:
            meta = sitk_meta(wpath)
            row.update({f'img_{k}': v for k, v in meta.items()})
        except Exception as e:
            row['img_shape'] = f'ERROR: {e}'

        # Fat-fraction image
        ffpath = wpath.replace('_WATER', '_FATFRACTION')
        if os.path.exists(ffpath):
            try:
                meta_ff = sitk_meta(ffpath)
                row.update({f'ff_{k}': v for k, v in meta_ff.items()})
            except Exception as e:
                row['ff_shape'] = f'ERROR: {e}'
        else:
            row['ff_shape'] = 'missing'

        # Ground truth mask
        gt_path = os.path.join(subj_dir, 'SegmentationMasks',
                               f'combined_gt_stack{stack_n}.mha')
        if os.path.exists(gt_path):
            try:
                meta_gt = sitk_meta(gt_path)
                row.update({f'gt_{k}': v for k, v in meta_gt.items()})
                row['img_gt_match'] = row.get('img_shape') == row.get('gt_shape')
            except Exception as e:
                row['gt_shape'] = f'ERROR: {e}'
        else:
            row['gt_shape'] = 'missing'

        rows.append(row)

df = pd.DataFrame(rows)
print(f'{len(df)} stacks across {df["subject"].nunique()} subjects')
df.head(3)

46 stacks across 21 subjects


,subject,type,stack,img_slices,img_height,img_width,img_shape,img_sp_x_mm,img_sp_y_mm,img_sp_z_mm,...,ff_sp_y_mm,ff_sp_z_mm,gt_slices,gt_height,gt_width,gt_shape,gt_sp_x_mm,gt_sp_y_mm,gt_sp_z_mm,img_gt_match
0,HV001_1,HV,1,65,672,672,"(65, 672, 672)",1.0,1.0,4.0,...,1.0,4.0,65,672,672,"(65, 672, 672)",1.0,1.0,4.0,True
1,HV001_1,HV,2,65,560,560,"(65, 560, 560)",1.0,1.0,4.0,...,1.0,4.0,65,560,560,"(65, 560, 560)",1.0,1.0,4.0,True
2,HV001_2,HV,1,65,672,672,"(65, 672, 672)",1.0,1.0,4.0,...,1.0,4.0,65,672,672,"(65, 672, 672)",1.0,1.0,4.0,True


In [3]:
# ── Summary: unique shape combinations ───────────────────────────────────────
summary = (
    df.groupby(['type', 'img_shape', 'gt_shape'])
    .agg(count=('subject', 'count'),
         subjects=('subject', lambda x: ', '.join(sorted(x.unique()))))
    .reset_index()
)
display(summary)

,type,img_shape,gt_shape,count,subjects
0,HV,"(65, 560, 560)","(65, 560, 560)",17,"HV001_1, HV001_2, HV001_3, HV002_1, HV002_2, H..."
1,HV,"(65, 672, 672)","(65, 672, 672)",17,"HV001_1, HV001_2, HV001_3, HV002_1, HV002_2, H..."
2,P,"(30, 512, 512)","(30, 512, 512)",12,"P001_1, P002_1, P003_1, P004_1"


In [ ]:
# ── Interactive per-subject browser ──────────────────────────────────────────
subject_dd = widgets.Dropdown(
    options=sorted(df['subject'].unique()),
    description='Subject:',
    layout=widgets.Layout(width='300px'),
)
out = widgets.Output()

def show_subject(change):
    subj = change['new']
    sub  = df[df['subject'] == subj].copy()
    cols = [c for c in sub.columns
            if any(c.startswith(p) for p in ('stack','img_shape','img_sp',
                                              'ff_shape','gt_shape','img_gt'))]
    with out:
        out.clear_output(wait=True)
        display(sub[['stack'] + [c for c in cols if c != 'stack']]
                .set_index('stack')
                .style.set_caption(subj))

subject_dd.observe(show_subject, names='value')
show_subject({'new': subject_dd.value})

display(subject_dd, out)

In [ ]:
# ── Flag any stacks where image and GT shapes do not match ────────────────────
mismatches = df[df['img_shape'] != df['gt_shape']]
if mismatches.empty:
    print('All image and GT shapes match.')
else:
    print(f'{len(mismatches)} shape mismatch(es):')
    display(mismatches[['subject', 'stack', 'img_shape', 'gt_shape']])